# ICESat-2 lidar: bathymetry (ATL24) and ground / canopy heights (ATL08)

Give a polygon of your area and get:

- **ATL24 seabed photons**, cleaned with transparent rules (every rejected photon keeps the reason), and split into
  **training** and **validation** tracks that are chosen once and **locked**, ready for Bathymetrix-AI
  (`ortho_h`, metres relative to EGM2008, negative below);
- **ATL08 / PhoREAL** ground elevation and canopy height per 100 m segment, e.g. over mangroves, with a verified
  conversion to EGM2008 heights.

ICESat-2 is downloaded with **SlideRule** (no account needed). No Earth Engine.

**Everything lives in one folder of your Google Drive**: download [ICESat2_tool.zip](https://github.com/Alerovere/seastrong-field-tools/releases/download/icesat2-tool-latest/ICESat2_tool.zip) (latest version), unzip it,
upload the `ICESat2_tool` folder anywhere in your Drive and open this notebook from there (double-click → *Open with
Google Colaboratory*). Results go to its `outputs` subfolder. Each step has an explanation cell, an options cell (every
option with its default and a comment; a misspelt name stops with a hint) and a run cell. Downloads are cached per year.

## Install (from your Drive folder)

Set `TOOLKIT_DIR` to the `ICESat2_tool` folder in your Drive. The cell mounts Drive, installs the packages it needs
(a *constraints file* keeps Colab's own Google packages unchanged) and imports the code from that folder.
`UPDATE_FROM_GITHUB = True` replaces the code by the latest release (old `src` backed up; `outputs` and this notebook
untouched); then *Runtime → Restart session*.

In [ ]:
TOOLKIT_DIR = '/content/drive/MyDrive/SEASTRONG/ICESat2_tool'   # the ICESat2_tool folder you saved in Drive
UPDATE_FROM_GITHUB = False

import subprocess, sys, re, os, glob
from google.colab import drive
drive.mount('/content/drive')
cands = [TOOLKIT_DIR, os.path.join(TOOLKIT_DIR, 'ICESat2_tool')] + sorted(glob.glob(os.path.join(TOOLKIT_DIR, '*', 'ICESat2_tool')))
found = [c for c in cands if os.path.exists(os.path.join(c, 'pyproject.toml')) and os.path.isdir(os.path.join(c, 'src', 'icesat2_tool'))]
if not found:
    raise FileNotFoundError(f'No ICESat2_tool folder (with pyproject.toml and src/icesat2_tool) in {TOOLKIT_DIR}. '
                            'In the Files panel on the left, right-click the folder -> Copy path.')
TOOLKIT_DIR = found[0]; print('toolkit folder:', TOOLKIT_DIR)

if UPDATE_FROM_GITHUB:
    import urllib.request, zipfile, shutil, io, time
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://github.com/Alerovere/seastrong-field-tools/releases/download/icesat2-tool-latest/ICESat2_tool.zip').read()))
    pre = [n for n in z.namelist() if n.endswith('pyproject.toml')][0][:-len('pyproject.toml')]
    backup = os.path.join(TOOLKIT_DIR, f"_backup_src_{time.strftime('%Y%m%d_%H%M%S')}")
    shutil.copytree(os.path.join(TOOLKIT_DIR, 'src'), backup)
    for n in z.namelist():
        rel = n[len(pre):]
        if not rel or n.endswith('/') or rel.startswith(('outputs/', '_backup')): continue
        if rel.startswith('notebooks/') and rel.endswith('.ipynb'): rel = rel[:-6] + '_NEW.ipynb'
        dst = os.path.join(TOOLKIT_DIR, rel); os.makedirs(os.path.dirname(dst), exist_ok=True)
        with open(dst, 'wb') as f: f.write(z.read(n))
    print(f'Code updated (old code in {backup}). Runtime -> Restart session, set UPDATE_FROM_GITHUB = False, run again.')

frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.splitlines()
protect = re.compile(r'^(google-|googleapis-|grpcio|protobuf|proto-plus|httplib2|ipykernel|ipython|jupyter|notebook|tornado|requests|urllib3)', re.I)
with open('/content/colab_constraints.txt', 'w') as f:
    f.write('\n'.join(l for l in frozen if '==' in l and protect.match(l)))
import tomllib
deps = tomllib.load(open(os.path.join(TOOLKIT_DIR, 'pyproject.toml'), 'rb'))['project']['dependencies']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps, '-c', '/content/colab_constraints.txt'], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-2500:])
if r.returncode != 0: raise RuntimeError('Installation failed (see above).')
sys.path.insert(0, os.path.join(TOOLKIT_DIR, 'src'))
OUTPUTS = os.path.join(TOOLKIT_DIR, 'outputs'); os.makedirs(OUTPUTS, exist_ok=True)
print('outputs folder:', OUTPUTS)

In [ ]:
import icesat2_tool as it
import tomllib
print('icesat2_tool', it.__version__, 'from', os.path.dirname(it.__file__))
_v = tomllib.load(open(os.path.join(TOOLKIT_DIR, 'pyproject.toml'), 'rb'))['project']['version']
if it.__version__ != _v or not it.__file__.startswith(TOOLKIT_DIR):
    raise RuntimeError(f'Version {it.__version__} is loaded but {TOOLKIT_DIR} has {_v}: Runtime -> Restart session and run again.')
# after editing the code in src/icesat2_tool: it.reload(), then run the cells from the site cell down

## Site

- **`AOI_FILE`**: a polygon of the area of interest in your Drive: GeoJSON, KML, shapefile (`.shp` with its
  companion files) or GeoPackage, any coordinate system. Draw it in QGIS or Google Earth (save as KML). In Colab's
  Files panel, right-click the file → *Copy path*.
- **`CRS`**: projected coordinate system for distances and maps; `None` picks the UTM zone of the AOI.
- **`NAME`**: start of every file name and the output subfolder.

In [ ]:
SITE = dict(
    NAME='Belize_south',          # file names and output subfolder
    AOI_FILE='/content/drive/MyDrive/SEASTRONG/aoi_belize_south.geojson',   # polygon of the area of interest
    CRS=None,                     # None = UTM zone of the AOI (e.g. 'EPSG:32616' for southern Belize)
    OUTPUT_DIR=OUTPUTS,
)
ctx = it.setup(SITE)

## Step 1 — ATL24 bathymetry: download, clean, lock the validation tracks

**What it does**, in an order that keeps validation honest (validation must not help decide the cleaning):
1. **Download** ATL24 (the official ICESat-2 bathymetry product) year by year over the AOI, with all quality fields.
2. **Independent checks** of each photon, using nothing from other tracks. Every rule writes its own flag column:

| Flag | Rejects a photon when |
|---|---|
| `f_missing` | a required value is missing |
| `f_class` | it is not classified as seabed (40) or its confidence < `MIN_CONFIDENCE` |
| `f_atl24` | ATL24 marks it unreliable (low confidence, invalid Kd, deeper than the water clarity allows) or its vertical uncertainty > `MAX_SIGMA_TVU` |
| `f_depth` | depth < `DEPTH_MIN` or > `DEPTH_MAX` |
| `f_land`, `f_shore` | it is on land or closer than `SHORE_BUFFER` m to it — **only with a `SHORELINE_FILE`** |
| `f_spike` | it jumps more than `SPIKE_MIN` m (or 3 × the local spread) from its neighbours along the track |

3. **Validation tracks**: whole satellite tracks (RGTs) go to validation by a simple rule (`rgt % m == k`), chosen
   once to give about `VALID_SHARE` of the photons with a similar depth range, and **locked** in a JSON file in the
   output folder. Later runs (e.g. with new data) re-use it, so validation tracks never move into training. Delete
   the lock file only to start a new hold-out on purpose, and say so in your methods.
4. **Cross-track check** (`f_xtrack`): each photon is compared with ≥ `XT_MIN_PASSES` *other training* passes within
   `XT_RADIUS` m. On validation tracks this flag is only informative (`XT_APPLY_TO_VALID=False`).
5. **Files**: `<NAME>_ATL24_all_with_QC_flags.gpkg` (every photon with all flags, to inspect in QGIS),
   `<NAME>_ATL24_train.gpkg` and `<NAME>_ATL24_valid.gpkg` (clean photons; Bathymetrix-AI depth field **`ortho_h`**),
   coverage tables, a figure and a README.

**Shoreline.** Without `SHORELINE_FILE`, photons on land or at the water's edge are removed only by ATL24's own
classification and the depth and spike rules. For reef islands, mangrove fringes or river mouths, give land polygons
(e.g. digitised in QGIS, or OpenStreetMap land) or coastline lines: distances are then exact.

**Datum.** `ortho_h` is relative to EGM2008, `depth` = water depth at the time of the pass; tides are not removed.

In [ ]:
ATL24 = dict(
    T0='2018-10-13',              # first day
    T1=None,                      # last day (None = today)
    SR_RETRIES=3,                 # attempts per year if SlideRule fails
    ON_MISSING_QC_FIELD='fail',   # 'fail' or 'quarantine' if ATL24 quality fields are absent
    MIN_CONFIDENCE=0.6,           # seabed confidence threshold
    MAX_SIGMA_TVU=1.0,            # m; total vertical uncertainty threshold
    DEPTH_MIN=0.3,                # m; shallower -> f_depth
    DEPTH_MAX=40.0,               # m; deeper -> f_depth
    SHORELINE_FILE=None,          # land polygons or coastline lines (GeoJSON / KML / shapefile / GeoPackage); None = not used
    SHORE_BUFFER=20.0,            # m from land -> f_shore
    SPIKE_WINDOW=15,              # photons in the along-track running median
    SPIKE_MIN=0.5,                # m
    VALID_SHARE=0.25,             # target share of photons on validation tracks
    MIN_RGT_TOTAL=4,              # fewer tracks -> no split
    MIN_RGT_VALID=1,
    MIN_PHOTONS_SIDE=50,
    HOLDOUT_LOCK=None,            # None = <outputs>/<NAME>/<NAME>_ICESat2_holdout_lock.json
    XT_RADIUS=30.0,               # m
    XT_MIN=1.0,                   # m
    XT_REL=0.20,                  # flag if deviation > max(XT_MIN, XT_REL x depth)
    XT_MIN_PASSES=3,              # other training passes needed
    XT_BIN=10.0,                  # m; along-track bins
    XT_APPLY_TO_VALID=False,      # True = also remove cross-track outliers from validation
)
it.atl24.run(ctx, ATL24)

## Step 2 — ATL08 ground and canopy heights (optional)

**What it does.** Downloads ATL08 / PhoREAL 100 m land segments: ground elevation and canopy height above ground.
With `MANGROVE_FILE` (e.g. Global Mangrove Watch polygons for your area, downloaded from globalmangrovewatch.org and
cut in QGIS) only segments inside the mangroves are kept; without it, every land segment in the AOI.

Ground heights come on the WGS84 ellipsoid and are converted to **EGM2008**; the conversion is **verified** (a real
geoid grid, not a zero-shift fallback). If it cannot be verified, `ground_h_egm08` stays empty and no segment is
certified. `DEM_FILES` (optional) compares DEMs in EGM2008 heights (e.g. FABDEM, DeltaDTM exported from your Earth
Engine script) with the ICESat-2 ground.

In [ ]:
ATL08 = dict(
    RUN=True,
    T0='2018-10-13',
    T1=None,
    SR_RETRIES=3,
    ATL08_LEN=100,                # m; segment length
    MANGROVE_FILE=None,           # mangrove polygons; None = all land segments in the AOI
    MANGROVE_BUFFER_M=30,         # m around the mangrove polygons
    MIN_GROUND_PH=5,              # ground photons needed per segment
    GROUND_RANGE=[-3.0, 6.0],     # m EGM2008; plausible ground elevations
    DEM_FILES={},                 # e.g. {'FABDEM': '/content/drive/MyDrive/.../fabdem.tif'} (EGM2008 heights)
)
it.atl08.run(ctx, ATL08)

## Using the results in Bathymetrix-AI

- Phase 2 (training): `<NAME>_ATL24_train.gpkg`, depth field `ortho_h` (negative below EGM2008).
- Phase 5 (validation): `<NAME>_ATL24_valid.gpkg` — never use it in the Phase 4 residual correction.
- Check in QGIS `<NAME>_ATL24_all_with_QC_flags.gpkg`, coloured by each `f_*` column, before training.
- Keep echosounder data referred to the same vertical datum as the final, independent check.